# Урок 12. Логические уравнения

10 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-11.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-13.ipynb)

---

Решение уравнений и систем логических уравнений. Подсчёт числа решений. Метод отображения и метод перебора. Проверка перебором в коде.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-12", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Что значит «решить логическое уравнение»

Обычное уравнение просит найти число. Логическое уравнение просит найти
**все наборы значений переменных**, при которых выражение истинно.

> Решите уравнение **(A ∨ B) ∧ ¬C = 1**.

Решением будет не одно значение, а список наборов: (0,1,0), (1,0,0),
(1,1,0). Чаще всего в ЕГЭ спрашивают даже не сами наборы, а только
**сколько их**.

Уравнение с n переменными имеет не больше 2ⁿ решений — столько всего
существует наборов. Для 5 переменных это 32 варианта, для 10 —
1024, а для 30 — уже миллиард, и перебор перестаёт помогать.

### Три способа решать

| Способ | Когда применим | Чем плох |
|---|---|---|
| перебор программой | до 20–25 переменных | на экзамене нет компьютера |
| таблица истинности | 2–3 переменные | 4 переменные — уже 16 строк |
| рассуждение | всегда | нужно думать |

На уроке будем делать и то и другое: сначала решать рассуждением,
потом проверять перебором. Так и на экзамене надёжнее — привычка
проверять себя стоит нескольких баллов.

### Приём 1. Конъюнкция = 1 фиксирует всё сразу

Если произведение равно единице, каждый множитель обязан быть
единицей. Если сумма равна нулю, каждое слагаемое обязано быть нулём.

```
(A ∨ B ∨ C) = 0     →  A=0, B=0, C=0        ровно одно решение
(A ∧ B ∧ C) = 1     →  A=1, B=1, C=1        ровно одно решение
```

Поэтому первое, что делают с уравнением, — приводят его к виду, где
одна часть жёстко задаёт значения переменных.

### Приём 2. Импликация ложна ровно в одном случае

```
(A → B) = 0   →   A=1, B=0        одно решение
(A → B) = 1   →   всё остальное — три решения
```

Это самый быстрый способ отсечь лишние варианты: уравнение
с импликацией и нулём справа сразу задаёт обе переменные.

### Приём 3. Цепочка импликаций

Классика ЕГЭ:

> Сколько решений у уравнения
> **(x₁ → x₂) ∧ (x₂ → x₃) ∧ (x₃ → x₄) = 1**?

Разберёмся без перебора. Импликация ложна только при переходе 1 → 0.
Значит, в наборе не должно быть места, где единица стоит перед нулём.
Такие наборы выглядят одинаково: сначала нули, потом только единицы.

```
0 0 0 0
0 0 0 1
0 0 1 1
0 1 1 1
1 1 1 1
```

Для цепочки из n переменных решений всегда **n + 1**: граница между
нулями и единицами может стоять в любом из n + 1 мест.

Запомните этот результат — он встречается в ЕГЭ почти каждый год,
обычно с дополнительными условиями.

### Приём 4. Системы уравнений

Система — несколько уравнений, которые должны выполняться
одновременно. Логически это просто конъюнкция всех уравнений:

```
⎧ (A → B) = 1
⎨                    равносильно     (A → B) ∧ (B → C) = 1
⎩ (B → C) = 1
```

Значит, решать систему — то же самое, что решать одно уравнение,
собранное из всех её частей знаком ∧. Каждое новое уравнение может
только уменьшить число решений или оставить его прежним.

## Смотрим, как это работает

### Пример 1. Перебор: все решения и их количество

In [ ]:
from itertools import product


def решения(функция, переменных):
    найденные = []
    for набор in product([0, 1], repeat=переменных):
        if функция(*набор):
            найденные.append(набор)
    return найденные


уравнение = lambda a, b, c: (a or b) and not c

все = решения(уравнение, 3)
print("Решений:", len(все))
for набор in все:
    print("  ", набор)

Функция `решения` пригодится во всех задачах урока: она перебирает
наборы в том же порядке, что и таблица истинности, и складывает
подходящие в список.

### Пример 2. Цепочка импликаций

In [ ]:
def импл(a, b):
    return (not a) or b


def цепочка(*x):
    for i in range(len(x) - 1):
        if not импл(x[i], x[i + 1]):
            return False
    return True


for сколько in range(2, 8):
    print(f"переменных: {сколько}  решений: {len(решения(цепочка, сколько))}")

Ровно n + 1, как и обещало рассуждение. Полезная привычка: если
формула в голове расходится с перебором, ошибка в рассуждении,
а не в компьютере.

Посмотрим на сами наборы для четырёх переменных:

In [ ]:
for набор in решения(цепочка, 4):
    print(набор)

Видна та самая структура: нули, потом единицы, и граница едет слева
направо.

### Пример 3. Уравнение с нулём справа

> **(A ∨ ¬B) ∧ (C → A) = 0**

Рассуждение: произведение равно нулю, когда хотя бы один множитель
ноль. Первый множитель ноль при A = 0 и B = 1. Второй — при C = 1
и A = 0. В обоих случаях A = 0.

In [ ]:
уравнение_ноль = lambda a, b, c: not ((a or not b) and импл(c, a))

for набор in решения(уравнение_ноль, 3):
    print(набор)
print("Всего решений:", len(решения(уравнение_ноль, 3)))

Все решения содержат A = 0 — как и подсказывало рассуждение.

### Пример 4. Система уравнений

In [ ]:
def система(a, b, c):
    первое = импл(a, b)
    второе = импл(b, c)
    третье = a or c
    return первое and второе and третье


найденные = решения(система, 3)
print("Решений системы:", len(найденные))
for набор in найденные:
    print("  A B C =", набор)

Цепочка из двух импликаций дала бы четыре набора, а третье условие
отсекло из них один — тот, где всё по нулям.

### Пример 5. Когда перебор уже не годится

In [ ]:
import time

for переменных in (10, 15, 20):
    начало = time.time()
    сколько = len(решения(цепочка, переменных))
    прошло = time.time() - начало
    print(f"{переменных} переменных: {2 ** переменных} наборов, "
          f"решений {сколько}, время {прошло:.2f} с")

Каждая новая переменная удваивает работу. Для 30 переменных перебор
занял бы часы, для 60 — дольше, чем существует Вселенная. Поэтому
в ЕГЭ такие задачи решают рассуждением: переменных там бывает
и сорок.

## Пробуем сами

### Задача 1. Сколько решений у импликации

Сколько решений у уравнения **(A → B) = 1**?

In [ ]:
#@title 🧩 Задача 1. Импликация { display-mode: "form" }
#@markdown Впишите число
решений_импл = 0 #@param {type:"integer"}

si.ответ("1", решений_импл, "4e07408562bedb8b",
         hint="Всего наборов четыре, а ложна импликация ровно в одном.")

### Задача 2. Цепочка из пяти

Сколько решений у уравнения
**(x₁ → x₂) ∧ (x₂ → x₃) ∧ (x₃ → x₄) ∧ (x₄ → x₅) = 1**?

In [ ]:
#@title 🧩 Задача 2. Цепочка из пяти { display-mode: "form" }
#@markdown Впишите число
решений_цепочки = 0 #@param {type:"integer"}

si.ответ("2", решений_цепочки, "e7f6c011776e8db7",
         hint="Для цепочки из n переменных решений n + 1.")

### Задача 3. Считаем решения программой

Напишите функцию: получает логическую функцию и количество переменных,
возвращает число наборов, при которых она истинна.

In [ ]:
def сколько_решений(функция, переменных):
    return ...

In [ ]:
si.check("3", сколько_решений, [
    ((lambda a, b: (not a) or b, 2), 3),
    ((lambda a, b, c: (a or b) and not c, 3), 3),
    ((lambda a: a, 1), 1),
    ((lambda a, b, c, d: (a == b) and (c == d), 4), 4),
])

### Задача 4. Уравнение с тремя переменными

Сколько решений у уравнения **(A ∧ B) ∨ (¬A ∧ C) = 1**?
Сначала посчитайте рассуждением: разберите отдельно случаи A = 1
и A = 0.

In [ ]:
решений_4 = ...

print(решений_4)

In [ ]:
si.ответ("4", решений_4, "4b227777d4dd1fc6",
         hint="При A = 1 нужно B = 1, при A = 0 нужно C = 1 — в каждом случае одна переменная свободна.")

### Задача 5. Уравнение с отрицанием

Сколько решений у уравнения **¬(A → B) = 1**, то есть при скольких
наборах импликация ложна?

In [ ]:
#@title 🧩 Задача 5. Отрицание импликации { display-mode: "form" }
#@markdown Впишите число
решений_5 = 0 #@param {type:"integer"}

si.ответ("5", решений_5, "6b86b273ff34fce1",
         hint="Импликация ложна только при A = 1, B = 0.")

### Задача 6. Все решения списком

Функция возвращает список всех наборов-решений (кортежей) для функции
от **двух** переменных, в порядке перебора от (0, 0) до (1, 1).

In [ ]:
def найти_решения(функция):
    return ...

In [ ]:
si.check("6", найти_решения, [
    (lambda a, b: a and b, [(1, 1)]),
    (lambda a, b: (not a) or b, [(0, 0), (0, 1), (1, 1)]),
    (lambda a, b: a and not a, []),
])

### Задача 7. Система уравнений

Сколько решений у системы?

```
⎧ (A → B) = 1
⎨ (B → C) = 1
⎩ (A ∨ C) = 1
```

In [ ]:
решений_системы = ...

print(решений_системы)

In [ ]:
si.ответ("7", решений_системы, "4e07408562bedb8b",
         hint="Цепочка даёт четыре набора, третье условие убирает набор из одних нулей.")

## Домашнее задание

### Домашнее задание 1. Длинная цепочка

Сколько решений у цепочки импликаций из **девяти** переменных
**(x₁ → x₂) ∧ … ∧ (x₈ → x₉) = 1**?

In [ ]:
#@title 🏠 Домашнее 1. Девять переменных { display-mode: "form" }
#@markdown Впишите число
дз_цепочка = 0 #@param {type:"integer"}

si.ответ("дз1", дз_цепочка, "4a44dc15364204a8",
         hint="n + 1.")

### Домашнее задание 2. Решения системы

Напишите функцию, которая возвращает количество решений системы

```
⎧ (x₁ ≡ x₂) = 1
⎨
⎩ (x₂ ⊕ x₃) = 1
```

для трёх переменных. Функция вызывается без аргументов.

In [ ]:
def решений_дз():
    return ...

In [ ]:
si.check("дз2", решений_дз, [
    ((), 2),
])

### Домашнее задание 3. Рассуждением, без перебора

Решите в тетради, не запуская программу:

1. Сколько решений у **(A ∨ B) ∧ (¬A ∨ C) = 1**?
2. Сколько решений у **(A → B) ∧ (¬A → C) = 1**?
3. Сколько решений у цепочки из шести переменных, если дополнительно
   известно, что x₁ = 0?

Потом проверьте каждый ответ перебором и запишите, где рассуждение
разошлось с программой.

---

### Любопытно

Задача «есть ли у логического уравнения хотя бы одно решение»
называется SAT и знаменита тем, что стала первой доказанно самой
трудной задачей своего класса. Быстрого способа решать её в общем
виде не найдено до сих пор, и за него полагается премия в миллион
долларов. При этом современные SAT-решатели прекрасно справляются
с уравнениями на миллионы переменных — просто потому, что реальные
задачи устроены не худшим образом.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-11.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-13.ipynb)